# PCA Plots from Excel Spreadsheet

Reads each dataset from its own tab in `PCA_datasets.xlsx` and produces a 2D PCA scatter plot for each one. To add or change a dataset, edit the spreadsheet tab (or add a new tab) and update the `DATASETS` list below — no other code needs to change.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## Spreadsheet location

Point this at the Excel file. Each dataset must live on its own sheet/tab, with the first column holding the row labels (product name, car name, student ID, city, etc.) and every other column holding the numeric variables used for PCA.

In [ ]:
EXCEL_PATH = "PCA_datasets.xlsx"

## Datasets to plot

Each entry maps a sheet name in the spreadsheet to the name of its label column (the non-numeric column identifying each row) and a plot title. Add or remove entries here to control which tabs get plotted.

In [ ]:
DATASETS = [
    {"sheet": "Products", "name_column": "Product",  "title": "PCA: Food Products"},
    {"sheet": "Cars",     "name_column": "Car",       "title": "PCA: Automobiles"},
    {"sheet": "Students", "name_column": "Student",   "title": "PCA: Students"},
    {"sheet": "Cities",   "name_column": "City",      "title": "PCA: Cities"},
]

## Principal Component Analysis

Consider the standardized data matrix $X \in \mathbb{R}^{n \times p}$ (n rows/observations, p numeric variables, each column mean-centered and scaled to unit variance). 


**Eigendecomposition of the covariance matrix.** The sample covariance matrix $\Sigma = \frac{1}{n-1}X^\top X$ is symmetric and positive semi-definite, so it admits an eigendecomposition:

$$
\Sigma = W \Lambda W^\top,
$$

where $W = [\,w_1 \; w_2 \; \cdots \; w_p\,]$ is an orthogonal matrix whose columns are the eigenvectors of $\Sigma$, and $\Lambda = \operatorname{diag}(\lambda_1, \lambda_2, \ldots, \lambda_p)$ is diagonal with $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_p \geq 0$. Each eigenvalue $\lambda_k$ equals the variance captured by $w_k$. 


**Recovering the PCA outputs from the decomposition.** 

- The **principal component directions** are the columns of $W$.
- The **component scores** (the coordinates plotted as PC1, PC2, ...) are $Z = XW $, i.e., projecting $X$ onto the columns of $W$.
- The **explained variance ratio** for component $k$ is $\lambda_k / \sum_{j=1}^p \lambda_j$.

**Why this matters in practice:**
- The decomposition view treats PCA as *factoring* $X$ (or $\Sigma$) into orthogonal pieces.
- Truncating to the first two components ($k=2$) for the scatter plots below is just keeping the leading two columns of $W$ and discarding the rest.


## Find principal component by solving optimization problem

Each principal component is a direction in $\mathbb{R}^p$ that a unit vector $w$ points along, and the component scores are the projections $z = Xw$.

**First principal component.** $w_1$ is the unit vector that maximizes the variance of the projected data:

$$
w_1 = \arg\max_{\lVert w \rVert = 1} \; \operatorname{Var}(Xw) = \arg\max_{\lVert w \rVert = 1} \; w^\top \Sigma \, w,
$$

In words: among all directions you could project the data onto, $w_1$ is the one that spreads the points out as much as possible — the direction of greatest variance.

**Second principal component.** $w_2$ solves the *same* maximization problem, but with an added orthogonality constraint that ties it back to the first solution:

$$
w_2 = \arg\max_{\lVert w \rVert = 1} \; w^\top \Sigma \, w \quad \text{subject to} \quad w \perp w_1.
$$

That is, $w_2$ captures as much of the *remaining* variance as possible, restricted to directions orthogonal (uncorrelated) to $w_1$.

**k-th principal component (general case).** This continues sequentially: each new component maximizes variance subject to being orthogonal to every component found before it:

$$
w_k = \arg\max_{\lVert w \rVert = 1} \; w^\top \Sigma \, w \quad \text{subject to} \quad w \perp w_1, w_2, \ldots, w_{k-1}.
$$

**Why this matters in practice:**
- The constraints are what make the components *sequential*: each one solves an optimization problem only after the previous ones are fixed, so PC2 can never "reuse" variance already explained by PC1.
- Because the components are mutually orthogonal, the total variance in the data splits additively across them — this is what `pca.explained_variance_ratio_` reports for each PC below.


In [ ]:
def make_pca_plot(df, name_column, title):

    labels = df[name_column]

    # Select numerical variables
    X = df.drop(columns=[name_column])

    # Standardize variables
    X_scaled = StandardScaler().fit_transform(X)

    # PCA -> 2 dimensions
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    # Explained variance
    print("\n", title)
    print("Explained variance:")
    print("PC1:", round(pca.explained_variance_ratio_[0] * 100, 2), "%")
    print("PC2:", round(pca.explained_variance_ratio_[1] * 100, 2), "%")

    # Plot
    plt.figure(figsize=(8, 6))
    plt.scatter(X_pca[:, 0], X_pca[:, 1],
                s=80, color="steelblue")

    for i, label in enumerate(labels):
        plt.annotate(label,
                     (X_pca[i, 0], X_pca[i, 1]),
                     xytext=(5, 5),
                     textcoords="offset points",
                     fontsize=10)

    plt.axhline(0, color="gray", linewidth=0.7)
    plt.axvline(0, color="gray", linewidth=0.7)

    plt.xlabel(
        f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)"
    )
    plt.ylabel(
        f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)"
    )
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## Read each tab from the spreadsheet and plot it

In [ ]:
for ds in DATASETS:
    df = pd.read_excel(EXCEL_PATH, sheet_name=ds["sheet"])
    make_pca_plot(df, ds["name_column"], ds["title"])

In [ ]:
#TODO:Answer questions:
# what happens to principal components if a feature has zero correlation with other features